# Spyglass access tutorial

This tutorial demonstrates how to query data for a Pagan Lab session from the Spyglass database.

The example session (`sub-P131_ses-TaskSwitch6-190815a`) contains behavioral data recorded with the BControl system. After running `insert_session()`, the following tables are populated:

1. **Connect** to the Spyglass database using DataJoint
2. **Core Spyglass tables** — `Nwbfile`, `Session`, `Subject`, `Lab`, `Institution`, `LabMember`, `IntervalList`
3. **Task metadata** — `TaskRecordingTypes`: state types, event types, action types, task arguments
4. **Behavioral data** — `TaskRecording`: per-trial events, states, actions, and trial parameters

For visualizations of the NWB data see the [NWB example notebook](https://github.com/catalystneuro/pagan-lab-to-nwb/blob/main/src/pagan_lab_to_nwb/tutorials/arc_behavior_dandi_demo_notebook.ipynb).

In [17]:
from pathlib import Path
import pandas as pd

DJ_LOCAL_CONF_PATH = Path(
    Path(__file__).parents[2] / "arc_ecephys" / "dj_local_conf.json"
)

import datajoint as dj
dj.config.load(str(DJ_LOCAL_CONF_PATH))
dj.conn(use_tls=False)

from spyglass.settings import raw_dir

nwb_file_name = "sub-P131_ses-TaskSwitch6-190815a.nwb"
nwbfile_path = Path(raw_dir) / nwb_file_name


[2026-04-15 14:00:24,509][INFO]: DataJoint is configured from /Users/weian/catalystneuro/pagan-lab-to-nwb/src/pagan_lab_to_nwb/spyglass_mock/dj_local_conf.json


In [18]:
from spyglass.utils.nwb_helper_fn import get_nwb_file, get_nwb_copy_filename

nwbfile = get_nwb_file(str(nwbfile_path))
nwb_copy_file_name = get_nwb_copy_filename(nwbfile_path.name)
nwb_dict = dict(nwb_file_name=nwb_copy_file_name)

print("NWB file:     ", nwbfile_path)
print("Spyglass copy:", nwb_copy_file_name)


NWB file:      /Volumes/T9/data/Pagan/raw/sub-P131_ses-TaskSwitch6-190815a.nwb
Spyglass copy: sub-P131_ses-TaskSwitch6-190815a_.nwb


## Core Spyglass tables

When `insert_session()` is called, Spyglass populates a set of common tables from the NWB file metadata.
The tables below reflect everything that was inserted for this session.

### Nwbfile

The `Nwbfile` table is the root entry — every other table is keyed by `nwb_file_name` (the Spyglass copy name, ending in `_`).

In [19]:
from spyglass.common import Nwbfile
import spyglass.common as sgc

nwbfile_entry = (Nwbfile() & nwb_dict).fetch(as_dict=True)
pd.DataFrame(nwbfile_entry)[["nwb_file_name", "nwb_file_abs_path"]]

,nwb_file_name,nwb_file_abs_path
0,sub-P131_ses-TaskSwitch6-190815a_.nwb,/Volumes/T9/data/Pagan/raw/sub-P131_ses-TaskSw...


### Session

`Session` stores the per-session metadata extracted from the NWB file: institution, lab, session description, timestamps, and the list of experimenters.

In [20]:
session = (sgc.Session() & nwb_dict).fetch(as_dict=True)
session_df = pd.DataFrame(session)
cols = ["nwb_file_name", "session_id", "institution_name", "lab_name",
        "session_description", "session_start_time", "timestamps_reference_time"]
session_df[cols]

,nwb_file_name,session_id,institution_name,lab_name,session_description,session_start_time,timestamps_reference_time
0,sub-P131_ses-TaskSwitch6-190815a_.nwb,TaskSwitch6-190815a,University of Edinburgh,Pagan,This session contains behavioral data from a r...,2019-08-15 11:41:00,2019-08-15 11:41:00


In [21]:
# Experimenters are stored as a separate part-table joined to Session
experimenters = (sgc.Session.Experimenter() & nwb_dict).fetch(as_dict=True)
pd.DataFrame(experimenters)

,nwb_file_name,lab_member_name
0,sub-P131_ses-TaskSwitch6-190815a_.nwb,Marino Pagan
1,sub-P131_ses-TaskSwitch6-190815a_.nwb,Vincent D. Tang


### Subject

`Subject` holds the animal metadata: species, sex, date of birth, and description.

In [22]:
# Subject is keyed by subject_id; join with Session to get the subject for this session
subject_id = session_df["subject_id"].iloc[0]
subject = (sgc.Subject() & {"subject_id": subject_id}).fetch(as_dict=True)
pd.DataFrame(subject)[["subject_id", "species", "sex", "age", "description"]]

,subject_id,species,sex,age,description
0,P131,Rattus norvegicus,M,P6M/P2Y,All subjects were male Long-Evans rats between...


### Lab and Institution

`Lab` and `Institution` are lookup tables populated from the NWB file's lab and institution fields.
`LabMember` stores each experimenter listed in the NWB file.

In [23]:
lab_name = session_df["lab_name"].iloc[0]
institution_name = session_df["institution_name"].iloc[0]

print("=== Lab ===")
display(pd.DataFrame((sgc.Lab() & {"lab_name": lab_name}).fetch(as_dict=True)))

print("=== Institution ===")
display(pd.DataFrame((sgc.Institution() & {"institution_name": institution_name}).fetch(as_dict=True)))

print("=== LabMember ===")
# LabMember rows for experimenters on this session
experimenter_names = [row["lab_member_name"] for row in experimenters]
lab_members = (sgc.LabMember() & [{"lab_member_name": n} for n in experimenter_names]).fetch(as_dict=True)
display(pd.DataFrame(lab_members))

=== Lab ===


,lab_name
0,Pagan


=== Institution ===


,institution_name
0,University of Edinburgh


=== LabMember ===


,lab_member_name,first_name,last_name
0,Marino Pagan,Marino,Pagan
1,Vincent D. Tang,Vincent D.,Tang


### IntervalList

`IntervalList` stores named time intervals associated with the session (e.g. valid recording periods). For behavior-only sessions this is empty — Spyglass populates it from NWB epochs, which arc_behavior files do not include.

In [24]:
interval_list = (sgc.IntervalList() & nwb_dict).fetch(as_dict=True)
pd.DataFrame(interval_list) if interval_list else print("(no IntervalList entries for behavior-only sessions)")

(no IntervalList entries for behavior-only sessions)


## Accessing the task metadata


The task-related general metadata is in `TaskRecordingTypes`.

The `StateTypes` table stores the type of states that occur during the task (e.g. while the animal is waiting for reward), one type per row.

This table is stored in `TaskRecordingTypes.StateTypes()`.


In [25]:
from spyglass.common.common_task_rec import TaskRecordingTypes

print("=== TaskRecordingTypes.StateTypes ===")
state_types = (TaskRecordingTypes.StateTypes() & nwb_dict).fetch(as_dict=True)
state_types_df = pd.DataFrame(state_types).set_index("id")
state_types_df

=== TaskRecordingTypes.StateTypes ===


,nwb_file_name,state_name
id,,
0,sub-P131_ses-TaskSwitch6-190815a_.nwb,state_0
1,sub-P131_ses-TaskSwitch6-190815a_.nwb,sending_trialnum
2,sub-P131_ses-TaskSwitch6-190815a_.nwb,check_next_trial_ready
3,sub-P131_ses-TaskSwitch6-190815a_.nwb,wait_for_cpoke
4,sub-P131_ses-TaskSwitch6-190815a_.nwb,wait_for_cpoke_wait2
5,sub-P131_ses-TaskSwitch6-190815a_.nwb,wait_for_cpoke_dir
6,sub-P131_ses-TaskSwitch6-190815a_.nwb,wait_for_cpoke_freq
7,sub-P131_ses-TaskSwitch6-190815a_.nwb,wait_for_cpoke_bis
8,sub-P131_ses-TaskSwitch6-190815a_.nwb,nic_prestim


The `EventTypes` is a column-based table to store the type of events that occur during the task (e.g. port poke from the animal), one type per row.

This table is stored in `TaskRecordingTypes.EventTypes()`.

In [26]:
event_types = (TaskRecordingTypes.EventTypes() & nwb_dict).fetch(as_dict=True)
event_types_df = pd.DataFrame(event_types).set_index("id")
print("=== TaskRecordingTypes.EventTypes ===")
event_types_df

=== TaskRecordingTypes.EventTypes ===


,nwb_file_name,event_name
id,,
0,sub-P131_ses-TaskSwitch6-190815a_.nwb,C
1,sub-P131_ses-TaskSwitch6-190815a_.nwb,L
2,sub-P131_ses-TaskSwitch6-190815a_.nwb,R


The `ActionTypes` is a column-based table to store the type of actions that occur during the task (e.g. sound output from the acquisition system), one type per row.

This table is stored in `TaskRecordingTypes.ActionTypes()`.

In [27]:
action_types = (TaskRecordingTypes.ActionTypes() & nwb_dict).fetch(as_dict=True)
action_types_df = pd.DataFrame(action_types).set_index("id")
print("=== TaskRecordingTypes.ActionTypes ===")
action_types_df

=== TaskRecordingTypes.ActionTypes ===


,nwb_file_name,action_name
id,,
0,sub-P131_ses-TaskSwitch6-190815a_.nwb,direct_reward
1,sub-P131_ses-TaskSwitch6-190815a_.nwb,stimulatorwave
2,sub-P131_ses-TaskSwitch6-190815a_.nwb,cpoke_timer


The arguments of the task is stored in an `Arguments` table which can be accessed as `TaskRecordingTypes.Arguments()`.


In [28]:
arguments = (TaskRecordingTypes.Arguments() & nwb_dict).fetch(as_dict=True)
arguments_df = pd.DataFrame(arguments)
print("\n=== TaskRecordingTypes.Arguments ===")
arguments_df[["nwb_file_name", "argument_name", "argument_description", "expression"]].head(20)


=== TaskRecordingTypes.Arguments ===


,nwb_file_name,argument_name,argument_description,expression
0,sub-P131_ses-TaskSwitch6-190815a_.nwb,CommentsSection_CommentsShow,Show/Hide Comments panel,0
1,sub-P131_ses-TaskSwitch6-190815a_.nwb,HistorySection_correct_coherent,Fraction of correct responses on coherent tria...,0.8222222222222222
2,sub-P131_ses-TaskSwitch6-190815a_.nwb,HistorySection_correct_incoherent,Fraction of correct responses on incoherent tr...,0.6983240223463687
3,sub-P131_ses-TaskSwitch6-190815a_.nwb,HistorySection_dir_correct,Fraction of correct responses on Direction (LO...,0.7206703910614525
4,sub-P131_ses-TaskSwitch6-190815a_.nwb,HistorySection_freq_correct,Fraction of correct responses on Frequency (FR...,0.7925925925925926
5,sub-P131_ses-TaskSwitch6-190815a_.nwb,HistorySection_hit_history_task,Binary trial outcomes within the current task ...,[ 0. 0. nan nan 1. nan nan nan nan nan nan ...
6,sub-P131_ses-TaskSwitch6-190815a_.nwb,HistorySection_incoh_history_task,Whether each trial in the current task block w...,[ 1. 1. nan nan 1. nan nan nan nan nan nan ...
7,sub-P131_ses-TaskSwitch6-190815a_.nwb,HistorySection_last_result,no description,correct
8,sub-P131_ses-TaskSwitch6-190815a_.nwb,HistorySection_left_correct,Fraction of correct responses on trials where ...,0.7290322580645161
9,sub-P131_ses-TaskSwitch6-190815a_.nwb,HistorySection_nTrials,no description,617


## Accessing the behavioral data

The `TaskRecording` stores the object references for events, states, and actions that occurred during the task.


In [29]:
from spyglass.common.common_task_rec import TaskRecording

task_recording = (TaskRecording() & nwb_dict).fetch(as_dict=True)

print("\n=== TaskRecording ===")
task_recording_df = pd.DataFrame(task_recording)
task_recording_df


=== TaskRecording ===


,nwb_file_name,actions_object_id,events_object_id,states_object_id,trials_object_id
0,sub-P131_ses-TaskSwitch6-190815a_.nwb,f8dcd5da-4008-439b-93ca-501a8a16db35,17b77ca1-44e2-4291-8ea5-bcebfcbe6da5,f6005b13-8cc6-46da-86c3-e69dfb779a9d,b9bc29fe-a111-404a-a780-cbdef20e0ab0


### Actions

In [30]:
actions_df = (TaskRecording() & nwb_dict).fetch1_dataframe("actions")
actions_df["action_type"] = actions_df["action_type"].apply(lambda x: x.values[0][0])
actions_df.head()

,timestamp,action_type,value,duration
id,,,,
0,1440.939255,stimulatorwave,out,0.010003
1,1440.949754,stimulatorwave,out,0.009506
2,1442.339263,cpoke_timer,out,0.000000
3,1449.867254,stimulatorwave,out,0.009499
4,1449.877256,stimulatorwave,out,0.010003


### Events

In [31]:
events_df = (TaskRecording() & nwb_dict).fetch1_dataframe("events")
events_df["event_type"] = events_df["event_type"].apply(lambda x: x.values[0][0])
events_df.head()

,timestamp,event_type,value,duration
id,,,,
0,1429.767753,R,out,1.741500
1,1431.802253,R,out,0.127001
2,1432.380754,R,out,0.114501
3,1440.148253,L,out,0.194001
4,1440.937753,C,out,0.065001


### States

In [32]:
states_df = (TaskRecording() & nwb_dict).fetch1_dataframe("states")
states_df["state_type"] = states_df["state_type"].apply(lambda x: x.values[0][0])
states_df.head()

,start_time,stop_time,state_type
id,,,
0,1426.550753,1445.614254,state_0
1,1426.550753,1426.566753,sending_trialnum
2,1426.566753,1427.566753,wait_for_cpoke
3,1427.566753,1427.567259,wait_for_cpoke_wait2
4,1427.567259,1428.567753,wait_for_cpoke_dir


## Trials

The Trials table contains one row per trial and includes task-related variables that may change over time; it also links each trial to the states, events, and actions that occurred during that trial.

In [33]:
trials_df = (TaskRecording() & nwb_dict).fetch1_dataframe("trials")
# we'll exclude the states, events, and actions columns for better display
columns_to_show = [col for col in trials_df.columns if col not in ("states", "events", "actions")]
trials_df[columns_to_show].head()

,start_time,stop_time,cpoke_start_time,left_hi,right_hi,left_lo,right_lo,crosstalk_dir,crosstalk_freq,bup_width,...,HistorySection_quadrant_history,HistorySection_task_history,HistorySection_incoh_history,HistorySection_gammadir_history,HistorySection_gammafreq_history,HistorySection_result_history,HistorySection_opto_left_history,HistorySection_opto_right_history,OptoSection_opto_connected,OptoSection_opto_type
id,,,,,,,,,,,,,,,,,,,,,
0,1426.550753,1445.614254,1440.938754,[],"[0.01842, 0.07042, 0.420615]",[],"[0.00171, 0.037, 0.04252, 0.04527, 0.08372, 0....",0,0,5,...,1,d,1,4.0,-2.5,3,0,0,0,Full Trial
1,1445.614753,1455.668754,1449.866753,[],"[0.02414, 0.21247, 0.287775, 0.293555, 0.54593...","[0.504595, 0.551575, 0.87734, 1.00831, 1.15587]","[0.035055, 0.057625, 0.062735, 0.132995, 0.148...",0,0,5,...,1,d,1,2.5,-1.0,1,0,0,1,Full Trial
2,1455.669255,1462.974753,1457.688753,"[0.08778, 0.10681, 0.300405, 0.397145, 0.43750...","[0.005325, 0.04738, 0.05887, 0.059205, 0.06422...",[],[],0,0,5,...,4,d,0,1.0,4.0,3,1020,1020,1,First Half
3,1462.975259,1473.800254,1468.262754,"[0.278885, 0.296635]",[],"[0.017075, 0.03374, 0.04821, 0.049495, 0.10852...","[0.334955, 1.077265]",0,0,5,...,2,d,0,-4.0,-4.0,3,0,0,1,First Half
4,1473.800753,1481.767753,1475.819754,"[0.15262, 0.21278, 0.238475, 0.261195, 0.29171...",[],"[0.06016, 0.102655, 0.11773, 0.176895, 0.18359...",[],0,0,5,...,2,d,0,-4.0,-1.0,1,0,0,1,First Half


In [34]:
# Get the first trial's states
trials_df.states[0]

,start_time,stop_time,state_type
id,,,
0,1426.550753,1445.614254,state_name id 0 state_0
1,1426.550753,1426.566753,state_name id 1 ...
2,1426.566753,1427.566753,state_name id 3 wait...
3,1427.566753,1427.567259,state_name id ...
4,1427.567259,1428.567753,state_name id ...
5,1428.567753,1440.937753,state_name id ...
6,1440.937753,1440.938754,state_name id 8 nic_prestim
7,1440.938754,1440.939255,state_name id 9 cpoke
8,1440.939255,1441.002754,state_name id 10 cpoke_in


In [35]:
# Get the first trial's events
trials_df.events[0]

,timestamp,event_type,value,duration
id,,,,
0,1429.767753,event_name id 2 R,out,1.741500
1,1431.802253,event_name id 2 R,out,0.127001
2,1432.380754,event_name id 2 R,out,0.114501
3,1440.148253,event_name id 1 L,out,0.194001
4,1440.937753,event_name id 0 C,out,0.065001
5,1441.300253,event_name id 0 C,out,0.038001


In [36]:
# Get the first trial's actions
trials_df.actions[0]

,timestamp,action_type,value,duration
id,,,,
0,1440.939255,action_name id 1 stim...,out,0.010003
1,1440.949754,action_name id 1 stim...,out,0.009506
2,1442.339263,action_name id 2 cpoke_timer,out,0.000000
